In [1]:
#Import
!pip install linearmodels
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.iv import IV2SLS


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load master panel
dataset = pd.read_csv(r'C:\Users\Spandan Dutta\econometrics_project\data\processed\master_panel.csv',parse_dates=['date'],index_col='date')
dataset.head()

,yield_1y,yield_2y,yield_3y,yield_4y,yield_5y,yield_6y,yield_7y,yield_8y,yield_9y,yield_10y,...,d_yield_30y,d_slope_10y_2y,d_fpi,d_us_10y,d_vix,d_brent,d_dxy,d_cpi_inflation,d_laf_net,d_repo_rate
date,,,,,,,,,,,,,,,,,,,,,
2021-04-30,3.80380,4.25790,4.89680,5.39230,5.68390,6.03340,6.21780,6.36770,6.13840,6.35670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-05-31,3.90921,4.36907,4.85644,5.26522,5.65988,6.08705,6.24015,6.51243,6.20216,6.25843,...,0.25824,-0.20944,-0.028150,-0.07,2.344310,3.727711,-1.3263,NaN,81460.818925,0.0
2021-06-30,4.05553,4.57015,5.12149,5.61436,5.80350,6.16466,6.34695,6.45004,6.29422,6.39280,...,0.08284,-0.06671,-0.028150,-0.13,-2.803682,4.629880,1.7957,NaN,-50881.678925,0.0
2021-07-31,3.81018,4.34746,4.94329,5.48950,5.83545,6.15497,6.38605,6.44795,6.37177,6.22949,...,0.06602,0.05938,0.005549,-0.21,0.646515,2.001818,0.1181,NaN,-56849.254624,0.0
2021-08-31,3.68540,4.22070,4.79860,5.37100,5.75670,6.01390,6.18990,6.25490,6.23740,6.37140,...,-0.07530,0.26867,0.005549,0.06,-0.130606,-4.418766,0.4362,NaN,-74986.290323,0.0


In [4]:
#regression 1
# Variables to be differenced
diff_cols = ['yield_2y', 'yield_5y', 'yield_10y', 'yield_30y','slope_10y_2y', 'slope_30y_10y','us_10y', 'vix', 'dxy', 'brent', 'laf_net']
#differencing
for col in diff_cols:
    dataset[f'd_{col}'] = dataset[col].diff()

# Controls 
controls = ['d_us_10y', 'd_vix', 'd_dxy', 'd_brent', 'repo_rate', 'cpi', 'd_laf_net']

# Regression dataset
reg_vars = ['d_slope_10y_2y','d_yield_10y','fpi','jpmorgan_weight'] + controls
df_reg = dataset[reg_vars].dropna()
print(df_reg.shape)

#OLS
Y=df_reg['d_slope_10y_2y']
X=sm.add_constant(df_reg[['fpi'] + controls])
model1 = sm.OLS(Y,X).fit()
model1.summary()

(47, 11)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         d_slope_10y_2y   R-squared:                       0.149
Model:                            OLS   Adj. R-squared:                 -0.030
Method:                 Least Squares   F-statistic:                    0.8339
Date:                Thu, 02 Apr 2026   Prob (F-statistic):              0.579
Time:                        18:37:11   Log-Likelihood:                 24.906
No. Observations:                  47   AIC:                            -31.81
Df Residuals:                      38   BIC:                            -15.16
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9433      1.282      0.736      0.466      -1.652       3.539
fpi            0.1028      0.089      1.160      0.253      -0.077       0.282
d_us_10y      -0.1209      0.115     -1.054      0.299      -0.353       0.111
d_vix         -0.0052      0.010     -0.544      0.590      -0.025       0.014
d_dxy          0.0125      0.017      0.738      0.465      -0.022       0.047
d_brent        0.0006      0.004      0.179      0.859      -0.007       0.008
repo_rate      0.0850      0.072      1.173      0.248      -0.062       0.232
cpi           -0.0112      0.012     -0.920      0.363      -0.036       0.013
d_laf_net  -3.253e-07   5.19e-07     -0.627      0.535   -1.38e-06    7.26e-07
==============================================================================
Omnibus:                       10.609   Durbin-Watson:                   2.191
Prob(Omnibus):                  0.005   Jarque-Bera (JB):               15.183
Skew:                          -0.649   Prob(JB):                     0.000505
Kurtosis:                       5.464   Cond. No.                     2.56e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.56e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [8]:
#IV regression 

#Stage 1 
X_stage1 = sm.add_constant(df_reg[['jpmorgan_weight'] + controls])
stage1 = sm.OLS(df_reg['fpi'], X_stage1).fit()
print(stage1.summary())


                            OLS Regression Results                            
Dep. Variable:                    fpi   R-squared:                       0.833
Model:                            OLS   Adj. R-squared:                  0.798
Method:                 Least Squares   F-statistic:                     23.69
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           1.57e-12
Time:                        18:44:45   Log-Likelihood:                 5.1318
No. Observations:                  47   AIC:                             7.736
Df Residuals:                      38   BIC:                             24.39
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -6.6460      1.675     

In [20]:
#Stage 2
Xc = sm.add_constant(df_reg[controls])
iv_slope = IV2SLS(df_reg['d_slope_10y_2y'],Xc,df_reg[['fpi']],instruments = df_reg[['jpmorgan_weight']]).fit(cov_type='robust')

iv_10y = IV2SLS(df_reg['d_yield_10y'],Xc,df_reg[['fpi']],instruments = df_reg[['jpmorgan_weight']]).fit(cov_type='robust')

print('2SLS slope:', round(iv_slope.params['fpi'], 4))
print('2SLS 10Y:', round(iv_10y.params['fpi'], 4))
print('OLS coef:', round(model1.params['fpi'], 4))
print('Difference:', round(abs(iv_slope.params['fpi'] - model1.params['fpi']), 4))

2SLS slope: 0.096
2SLS 10Y: -0.1446
OLS coef: 0.1028
Difference: 0.0068
